In [ ]:
import json
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
PROJECT_DIR = '/content/drive/MyDrive'

PROJECT_DIR = Path('/content/drive/MyDrive')

def load(name):
    with open(PROJECT_DIR / name) as f:
        return json.load(f)

emp        = load('audit_empirical_chance.json')
front      = load('audit_frontier_substitution.json')
shuffle    = load('audit_shuffle_summary.json')
det_mq     = load('detector_improved_validation.json')
det_mmc    = load('detector_medmcqa_validation.json')
retest     = load('audit_test_retest.json')
fivewrong  = load('audit_5wrong_baseline.json')
nondental  = load('audit_non_dental.json')
lifts      = load('audit_pairwise_lift_permutation.json')
mmc5       = load('medmcqa_5model_summary.json')
triang     = load('triangulation_summary.json')

print(f"{'='*72}\nPAPER NUMBERS — CANONICAL REFERENCE\n{'='*72}\n")

print(f"## Headline: 5-LLM unanimous convergence on MedQA")
print(f"  Subset (all 5 strong models wrong, MedQA): n = {front['all_5_strong_wrong_subset_size']}")
n_unanim_5 = round(front['unanimous_rate_5_strong'] * front['all_5_strong_wrong_subset_size'])
print(f"  Unanimous-wrong: {n_unanim_5}/{front['all_5_strong_wrong_subset_size']} = {front['unanimous_rate_5_strong']*100:.1f}%")
naive_5 = (1/3)**4
print(f"  Naive chance (1/3)^4: {naive_5*100:.2f}%")
print(f"  Naive ratio: {front['unanimous_rate_5_strong']/naive_5:.1f}x")

print(f"\n## 4-LLM convergence (empirical chance baselines)")
print(f"  MedQA:   observed {emp['medqa']['observed_rate']*100:.1f}%, "
      f"empirical chance {emp['medqa']['empirical_chance']*100:.2f}%, "
      f"ratio {emp['medqa']['honest_ratio']:.1f}x")
print(f"  MedMCQA: observed {emp['medmcqa']['observed_rate']*100:.1f}%, "
      f"empirical chance {emp['medmcqa']['empirical_chance']*100:.2f}%, "
      f"ratio {emp['medmcqa']['honest_ratio']:.1f}x")

print(f"\n## MedMCQA k-intersection z-scores (5 models)")
for k, z in mmc5['k_intersection_z_scores'].items():
    print(f"  {k}: z = {z:+.2f}")

print(f"\n## Pairwise failure lifts (permutation-tested)")
for ds_name, ds_key in [('MedQA', 'medqa'), ('MedMCQA', 'medmcqa')]:
    rows = lifts[ds_key]
    ss = [r for r in rows if r['kind'] == 'strong-strong']
    fs = [r for r in rows if r['kind'] == 'flan-strong']
    ss_lifts = [r['observed'] for r in ss]
    fs_lifts = [r['observed'] for r in fs]
    ss_z = [r['z'] for r in ss]
    fs_z = [r['z'] for r in fs]
    print(f"  {ds_name}:")
    print(f"    Strong-strong lifts: {min(ss_lifts):.2f}x – {max(ss_lifts):.2f}x (z {min(ss_z):.1f} to {max(ss_z):.1f})")
    print(f"    Flan-strong lifts:   {min(fs_lifts):.2f}x – {max(fs_lifts):.2f}x (z {min(fs_z):+.2f} to {max(fs_z):+.2f})")

print(f"\n## Mechanism test (distractor shuffle, mid-tier 3 models)")
print(f"  All 3 picked original-wrong TEXT:   {shuffle['mid_tier_3_joint']['all_3_picked_orig_wrong_text']*100:.1f}%")
print(f"  All 3 picked original-wrong LETTER: {shuffle['mid_tier_3_joint']['all_3_picked_orig_wrong_letter']*100:.1f}%")
print(f"  Chance baseline (text): {shuffle['mid_tier_3_joint']['expected_joint_text_under_indep']*100:.1f}%")
print(f"  Chance baseline (letter): {shuffle['mid_tier_3_joint']['expected_joint_letter_under_indep']*100:.1f}%")
print(f"  Content ratio: {shuffle['mid_tier_3_joint']['content_ratio_vs_chance']:.2f}x")
print(f"  Letter ratio:  {shuffle['mid_tier_3_joint']['letter_ratio_vs_chance']:.1f}x")
print(f"  GPT-4o caveat: recovered to {shuffle['per_model_under_shuffle']['gpt4o']['correct_after_shuffle']*100:.1f}% correct under shuffle")

print(f"\n## Test-retest (T=0 vs T=0.7 majority vote)")
print(f"  Llama: T=0 {retest['llama_acc_t0']*100:.1f}%, T=0.7 maj {retest['llama_acc_t07']*100:.1f}%")
print(f"  Qwen:  T=0 {retest['qwen_acc_t0']*100:.1f}%, T=0.7 maj {retest['qwen_acc_t07']*100:.1f}%")
print(f"  Unanim-wrong: T=0 n={retest['t0_unanim_count']}, T=0.7 n={retest['t07_unanim_count']}, overlap={retest['unanim_overlap']}")
print(f"  Unanim Jaccard: {retest['unanim_jaccard']:.3f}")
print(f"  Subset Jaccard: {retest['subset_jaccard']:.3f}")

print(f"\n## Non-Dental MedMCQA stratification")
print(f"  n_total = {nondental['n_total']}, n_non_dental = {nondental['n_non_dental']}")
print(f"  Unanimous rate non-dental: {nondental['unanimous_rate_non_dental']*100:.1f}%")
print(f"  Unanimous rate full:       {nondental['unanimous_rate_full']*100:.1f}%")

print(f"\n## Question-level vs capable-LLM-specific (5-wrong baseline)")
for ds_name, ds_key in [('MedQA', 'medqa'), ('MedMCQA', 'medmcqa')]:
    d = fivewrong[ds_key]
    print(f"  {ds_name}:")
    print(f"    Full subset:  {d['full']['unanim']}/{d['full']['n']} = {d['full']['rate']*100:.1f}%")
    print(f"    All-5-wrong:  {d['all_5_wrong']['unanim']}/{d['all_5_wrong']['n']} = {d['all_5_wrong']['rate']*100:.1f}%")
    print(f"    Trap (Flan right): {d['trap']['unanim']}/{d['trap']['n']} = {d['trap']['rate']*100:.1f}%")

print(f"\n## Bias classification (4 corners: dataset x classifier)")
for corner, vals in triang['four_corner_distribution'].items():
    print(f"  {corner:<22} PC={vals['PC']*100:.1f}%, ANCHOR={vals['ANCHOR']*100:.1f}%, n={vals['n']}")
print(f"  Cross-classifier kappa MedQA:   {triang['cross_classifier_agreement']['medqa_kappa']:.3f}")
print(f"  Cross-classifier kappa MedMCQA: {triang['cross_classifier_agreement']['medmcqa_kappa']:.3f}")

print(f"\n## Convergence-warning detector")
for ds_name, det in [('MedQA', det_mq), ('MedMCQA', det_mmc)]:
    hr = det['tier_results']['HIGH_RISK']
    cf = det['tier_results']['CONFIDENT']
    print(f"  {ds_name}:")
    print(f"    HIGH_RISK: n={hr['n']}, wrong rate {hr.get('mid_wrong_rate', hr.get('wrong_rate'))*100:.1f}%")
    print(f"    CONFIDENT: n={cf['n']}, wrong rate {cf.get('mid_wrong_rate', cf.get('wrong_rate'))*100:.1f}%")
    print(f"    Precision: {det['high_risk_precision']*100:.1f}%, Recall: {det['high_risk_recall']*100:.1f}%")

print(f"\n## Model accuracies")
print(f"  MedMCQA: " + ", ".join(f"{k} {v*100:.1f}%" for k, v in mmc5['model_accuracies'].items()))

paper_numbers = {
    'headline_5_llm': {
        'subset_n': front['all_5_strong_wrong_subset_size'],
        'unanimous_n': n_unanim_5,
        'unanimous_rate': front['unanimous_rate_5_strong'],
        'naive_chance': naive_5,
        'naive_ratio': front['unanimous_rate_5_strong']/naive_5,
    },
    'empirical_chance_4_model': emp,
    'shuffle_mechanism': shuffle,
    'test_retest': retest,
    'non_dental_stratification': nondental,
    'question_level_test': fivewrong,
    'pairwise_lifts': lifts,
    'medmcqa_5_model': mmc5,
    'bias_triangulation': triang,
    'detector_medqa': det_mq,
    'detector_medmcqa': det_mmc,
    'frontier_substitution': front,
}
with open(PROJECT_DIR / 'paper_numbers.json', 'w') as f:
    json.dump(paper_numbers, f, indent=2)
print(f"\n{'='*72}\nSaved consolidated paper_numbers.json — single source of truth\n{'='*72}")